Hay que tener instalado la versión de Java 17. Para ello, ejecutar en el terminal:
```bash
sudo apt install -y openjdk-17-jre openjdk-17-jdk
```

Cuando se ejecute en local, asegurarse de que usamos el kernel con el entorno virtual y en el terminal ejecutar:

```bash
pip install pyspark==4.0.0 requests pillow ipython
sudo apt install tree
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
```

In [1]:

!export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
!export SPARK_LOCAL_IP="192.168.0.20"

In [2]:
import requests
import zipfile
import os

# Define the URL of the zip file
zip_url = "https://api.data.neurosys.com:4443/agar-public/AGAR_demo.zip"

# Define the local filename to save the zip file
zip_filename = "AGAR_demo.zip"

# Define the directory to extract the contents into
extraction_directory = "AGAR_demo_extracted"

# Download the zip file
print(f"Downloading {zip_url}...")
try:
    response = requests.get(zip_url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(zip_filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Successfully downloaded {zip_filename}")

    # Unzip the file
    print(f"Extracting {zip_filename} to {extraction_directory}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extraction_directory)
    print(f"Successfully extracted contents to {extraction_directory}/")

    # Optional: Clean up the downloaded zip file
    # os.remove(zip_filename)
    # print(f"Removed downloaded zip file: {zip_filename}")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
except zipfile.BadZipFile:
    print(f"Error: The downloaded file '{zip_filename}' is not a valid zip file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Successfully downloaded AGAR_demo.zip
Extracting AGAR_demo.zip to AGAR_demo_extracted...
Successfully extracted contents to AGAR_demo_extracted/


In [3]:
!cd AGAR_demo_extracted && tree

.
└── AGAR_representative
    ├── higher-resolution
    │   ├── bright
    │   │   ├── 1399.jpg
    │   │   ├── 1399.json
    │   │   ├── 349.jpg
    │   │   ├── 349.json
    │   │   ├── 356.jpg
    │   │   ├── 356.json
    │   │   ├── 510.jpg
    │   │   ├── 510.json
    │   │   ├── 518.jpg
    │   │   ├── 518.json
    │   │   ├── 525.jpg
    │   │   ├── 525.json
    │   │   ├── 734.jpg
    │   │   ├── 734.json
    │   │   ├── 735.jpg
    │   │   ├── 735.json
    │   │   ├── 736.jpg
    │   │   ├── 736.json
    │   │   ├── 971.jpg
    │   │   └── 971.json
    │   ├── dark
    │   │   ├── 4826.jpg
    │   │   ├── 4826.json
    │   │   ├── 4904.jpg
    │   │   ├── 4904.json
    │   │   ├── 5206.jpg
    │   │   ├── 5206.json
    │   │   ├── 5207.jpg
    │   │   ├── 5207.json
    │   │   ├── 5212.jpg
    │   │   ├── 5212.json
    │   │   ├── 5270.jpg
    │   │   ├── 5270.json
    │   │   ├── 5271.jpg
    │   │   ├── 5271.json
    │   │   ├── 5308.jpg
    │   │   ├── 5308.json
    │   │   

# Paralelización

In [ ]:
import os
import json
import io
from PIL import Image

# Dimensiones objetivo para el dataset de segmentación
TARGET_WIDTH = 1024
TARGET_HEIGHT = 1024
ORIGINAL_WIDTH = 2048  # Son imágenes cuadradas
ORIGINAL_HEIGHT = 2048

SCALE_X = TARGET_WIDTH / ORIGINAL_WIDTH
SCALE_Y = TARGET_HEIGHT / ORIGINAL_HEIGHT

# Ruta de salida
# output_dir = "/mnt/shared_dataset/output"
output_dir = "/home/laptop/lab_spark/output"
os.makedirs(output_dir, exist_ok=True)

def logueo(mensaje):
    with open(os.path.join(output_dir, "log.txt"), "a") as log_file:
        log_file.write(f"{mensaje}\n")

def process_image_and_json(row):
    # 🌟 Forzamos la creación de la carpeta antes de abrir el archivo
    os.makedirs("/home/laptop/lab_spark/output", exist_ok=True)
    """
    Función que se ejecutará en los ordenadores de los alumnos (Workers).
    Recibe la ruta del JSON y busca su imagen emparejada.
    """
        
    json_path = row['path']
    # logueo(f"Procesando {row['path']}")

    # 🌟 CORRECCIÓN: Eliminar el protocolo 'file:' si Spark lo añade
    if json_path.startswith("file:"):
        json_path = json_path.replace("file:", "")
        
    # Si Spark añade un triple slash 'file:///home/...' tras quitarle 'file:', 
    # asegúrate de que no queden barras dobles al principio
    if json_path.startswith("//"):
        json_path = json_path[2:]
    # Reemplazar la extensión para obtener la imagen
    img_path = json_path.replace(".json", ".jpg")

    if not os.path.exists(img_path):
        return f"Error: Imagen no encontrada para {json_path}"

    try:
        # 1. Leer el JSON original
        with open(json_path, 'r') as f:
            data = json.load(f)

        # 2. Cargar la imagen original
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        img = Image.open(io.BytesIO(img_bytes))

        # --- TAREA 2 & 3: REESCALADO DE IMAGEN Y AJUSTE DE JSON ---
        img_resized = img.resize((TARGET_WIDTH, TARGET_HEIGHT))

        # Estructura para el nuevo JSON modificado
        new_labels = []
        for label in data.get("labels", []):
            # Recalcular las cajas de segmentación proporcionalmente
            new_x = int(label["x"] * SCALE_X)
            new_y = int(label["y"] * SCALE_Y)
            new_w = int(label["width"] * SCALE_X)
            new_h = int(label["height"] * SCALE_Y)

            new_labels.append({
                "id": label["id"],
                "class": label["class"],
                "x": new_x,
                "y": new_y,
                "width": new_w,
                "height": new_h
            })

            # --- TAREA 3: RECORTE (CROP) DE LA COLONIA ---
            # Coordenadas originales para el recorte de máxima calidad
            x, y, w, h = label["x"], label["y"], label["width"], label["height"]
            colony_crop = img.crop((x, y, x + w, y + h))

            # Definir ruta de salida del recorte por clase (Ej: /output/crops/B.subtilis/13895_1.jpg)
            crop_dir = f"{output_dir}/crops/{label['class']}"
            os.makedirs(crop_dir, exist_ok=True)
            crop_filename = f"{data['sample_id']}_{label['id']}.jpg"
            colony_crop.save(os.path.join(crop_dir, crop_filename))

        # Guardar imagen reescalada y nuevo JSON
        seg_img_dir = f"{output_dir}/segmentation/images"
        seg_json_dir = f"{output_dir}/segmentation/labels"
        os.makedirs(seg_img_dir, exist_ok=True)
        os.makedirs(seg_json_dir, exist_ok=True)

        filename_base = os.path.basename(json_path).replace(".json", "")
        img_resized.save(os.path.join(seg_img_dir, f"{filename_base}.jpg"))

        data["labels"] = new_labels
        with open(os.path.join(seg_json_dir, f"{filename_base}.json"), 'w') as f:
            json.dump(data, f, indent=4)

        return f"Procesado con éxito: {filename_base}"

    except Exception as e:
        return f"Error procesando {json_path}: {str(e)}"



In [5]:
import os
os.environ["SPARK_LOCAL_IP"] = "192.168.0.20"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://192.168.0.20:7077") \
    .config("spark.driver.host", "192.168.0.20") \
    .appName("BacteriaDatasetProcessing") \
    .getOrCreate()

print("¡Conectado con éxito al Clúster Distributed!")
# Buscamos todos los archivos .json del dataset
# (Asumiendo que el dataset está montado de forma compartida en la misma ruta para todos)
dataset_path = "AGAR_demo_extracted/AGAR_representative/"

# Leemos los paths usando Spark
json_files_df = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.json") \
    .option("recursiveFileLookup", "true") \
    .load(dataset_path) \
    .select("path")

# 🌟 DEBÚGUEO: Verifica cuántos archivos ha detectado Spark antes de procesar
print(f"Número de archivos JSON detectados por Spark: {json_files_df.count()}")

# Convertimos a RDD para procesar fila por fila de manera distribuida
results = json_files_df.rdd.map(process_image_and_json).collect()

# Imprimir resumen en el Driver
for res in results: # Muestra los primeros 10 resultados
    print(res)

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 17:23:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


¡Conectado con éxito al Clúster Distributed!


Número de archivos JSON detectados por Spark: 40


Procesado con éxito: 518
Procesado con éxito: 971
Procesado con éxito: 14581
Procesado con éxito: 5206
Procesado con éxito: 349
Procesado con éxito: 525
Procesado con éxito: 734
Procesado con éxito: 735
Procesado con éxito: 510
Procesado con éxito: 11924
Procesado con éxito: 12031
Procesado con éxito: 14684
Procesado con éxito: 11761
Procesado con éxito: 5271
Procesado con éxito: 5212
Procesado con éxito: 11764
Procesado con éxito: 12028
Procesado con éxito: 14618
Procesado con éxito: 14410
Procesado con éxito: 5270
Procesado con éxito: 14380
Procesado con éxito: 13938
Procesado con éxito: 11884
Procesado con éxito: 736
Procesado con éxito: 5312
Procesado con éxito: 12033
Procesado con éxito: 8386
Procesado con éxito: 4904
Procesado con éxito: 13895
Procesado con éxito: 14130
Procesado con éxito: 14512
Procesado con éxito: 14627
Procesado con éxito: 4826
Procesado con éxito: 5308
Procesado con éxito: 5207
Procesado con éxito: 11773
Procesado con éxito: 356
Procesado con éxito: 11955
Pr

Podemos comparar los tamaños de las imágenes

In [6]:
!ls "AGAR_demo_extracted/AGAR_representative/lower-resolution/" -lisa

total 6900
6162821   4 drwxrwxr-x 2 laptop laptop   4096 jun 21 13:45 .
6162756   4 drwxrwxr-x 4 laptop laptop   4096 jun 21 13:45 ..
6162822 652 -rw-rw-r-- 1 laptop laptop 664895 jun 21 17:23 13895.jpg
6162823   4 -rw-rw-r-- 1 laptop laptop   3471 jun 21 17:23 13895.json
6162824 620 -rw-rw-r-- 1 laptop laptop 634409 jun 21 17:23 13938.jpg
6162825   8 -rw-rw-r-- 1 laptop laptop   5685 jun 21 17:23 13938.json
6162826 720 -rw-rw-r-- 1 laptop laptop 733641 jun 21 17:23 14130.jpg
6162827   4 -rw-rw-r-- 1 laptop laptop   3337 jun 21 17:23 14130.json
6162828 648 -rw-rw-r-- 1 laptop laptop 662886 jun 21 17:23 14380.jpg
6162829   8 -rw-rw-r-- 1 laptop laptop   6096 jun 21 17:23 14380.json
6162830 724 -rw-rw-r-- 1 laptop laptop 737793 jun 21 17:23 14410.jpg
6162831   8 -rw-rw-r-- 1 laptop laptop   6485 jun 21 17:23 14410.json
6162832 716 -rw-rw-r-- 1 laptop laptop 732806 jun 21 17:23 14512.jpg
6162833   4 -rw-rw-r-- 1 laptop laptop   2804 jun 21 17:23 14512.json
6162834 736 -rw-rw-r-- 1 laptop 

In [7]:
!ls "AGAR_demo_extracted/AGAR_representative/higher-resolution/vague/" -lisa

total 32356
6162800    4 drwxrwxr-x 2 laptop laptop    4096 jun 21 13:45 .
6162757    4 drwxrwxr-x 5 laptop laptop    4096 jun 21 13:45 ..
6162801 3328 -rw-rw-r-- 1 laptop laptop 3407386 jun 21 17:23 11761.jpg
6162802   12 -rw-rw-r-- 1 laptop laptop    9491 jun 21 17:23 11761.json
6162803 3068 -rw-rw-r-- 1 laptop laptop 3141587 jun 21 17:23 11764.jpg
6162804    8 -rw-rw-r-- 1 laptop laptop    7303 jun 21 17:23 11764.json
6162805 3128 -rw-rw-r-- 1 laptop laptop 3201924 jun 21 17:23 11773.jpg
6162806    4 -rw-rw-r-- 1 laptop laptop    1013 jun 21 17:23 11773.json
6162807 3336 -rw-rw-r-- 1 laptop laptop 3414123 jun 21 17:23 11884.jpg
6162808    8 -rw-rw-r-- 1 laptop laptop    5475 jun 21 17:23 11884.json
6162809 3104 -rw-rw-r-- 1 laptop laptop 3177187 jun 21 17:23 11890.jpg
6162810    4 -rw-rw-r-- 1 laptop laptop     653 jun 21 17:23 11890.json
6162811 3468 -rw-rw-r-- 1 laptop laptop 3551134 jun 21 17:23 11924.jpg
6162812   12 -rw-rw-r-- 1 laptop laptop   10108 jun 21 17:23 11924.json
616

In [8]:
!ls "output/segmentation/images" -lisa

ls: no se puede acceder a 'output/segmentation/images': No existe el archivo o el directorio


Comprobamos que en el json, el recorte se ha reducido a la mitad.

In [9]:
!head -16 AGAR_demo_extracted/AGAR_representative/higher-resolution/vague/11761.json

{
    "background": "vague",
    "classes": [
        "P.aeruginosa",
        "S.aureus"
    ],
    "colonies_number": 54,
    "labels": [
        {
            "class": "S.aureus",
            "height": 39,
            "id": 1,
            "width": 39,
            "x": 706,
            "y": 2112
        },


In [10]:
!head -16 output/segmentation/labels/11761.json

head: no se puede abrir 'output/segmentation/labels/11761.json' para lectura: No existe el archivo o el directorio


In [11]:
# !rm -rf output